In [5]:
"""Site x month sampling heatmap -- standalone version: one row per
site, with `site`, `SiteCode`, `City`, and a `dates` column holding every
collection date for that site as a comma-separated ISO-date string.

One row per site (labeled by SiteCode, grouped by City -- most sites to
least, SiteCode order within a city), one column per calendar month, cell
shaded by how many distinct dates a sample was collected there that month.
"""

#Import required packages
import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap

SITE_METADATA_PATH = os.path.join("Data_1_Site_Metadata_Final.xlsx")
OUT_PATH = "fig1b_site_sample_dates_Final.png"

# Sequential ramp: near-white -> the project's blue (dataviz categorical slot 1),
# so this chart still reads as "the same blue" as other site-coverage figures.
CMAP = LinearSegmentedColormap.from_list("blue_seq", ["#eaf1fb", "#2a78d6", "#123a6b"])
CITY_LINE_COLOR = "#8a8878"
YEAR_LINE_COLOR = "#555555"


def load_site_dates():
    df = pd.read_excel(SITE_METADATA_PATH, sheet_name="Sheet2")
    df["SiteCode"] = df["SiteCode"].astype(str).str.strip()
    df["City"] = df["City"].astype(str).str.strip()

    rows = []
    for site_code, city, dates_str in zip(df["SiteCode"], df["City"], df["dates"]):
        for token in str(dates_str).split(","):
            token = token.strip()
            if token:
                rows.append((site_code, city, token))
    long_df = pd.DataFrame(rows, columns=["SiteCode", "City", "date"])
    long_df["date"] = pd.to_datetime(long_df["date"])
    return long_df


def main():
    site_dates = load_site_dates()
    site_dates["month"] = site_dates["date"].dt.to_period("M")

    site_city = site_dates.drop_duplicates("SiteCode")[["SiteCode", "City"]]
    first_date = site_dates.groupby("SiteCode")["date"].min().rename("first_date")
    site_city = site_city.merge(first_date, on="SiteCode")
    city_order = site_city["City"].value_counts()
    city_first_date = site_city.groupby("City")["first_date"].min()
    site_order = (
        site_city.assign(city_rank=lambda d: d["City"].map(city_order),
                          city_first_date=lambda d: d["City"].map(city_first_date))
        .sort_values(["city_rank", "city_first_date", "first_date"], ascending=[False, True, True])
    )
    sites = site_order["SiteCode"].tolist()
    site_to_city = dict(zip(site_order["SiteCode"], site_order["City"]))
    n_sites = len(sites)

    months = pd.period_range(site_dates["month"].min(), site_dates["month"].max(), freq="M")
    n_months = len(months)

    counts = (
        site_dates.groupby(["SiteCode", "month"]).size()
        .reindex(pd.MultiIndex.from_product([sites, months], names=["SiteCode", "month"]), fill_value=0)
        .unstack("month")
        .loc[sites, months]
    )
    data = np.ma.masked_equal(counts.to_numpy(), 0)

    x_edges = mdates.date2num(months.to_timestamp().tolist() + [(months[-1] + 1).to_timestamp()])
    y_edges = np.arange(n_sites + 1) - 0.5

    fig, ax = plt.subplots(figsize=(max(18, n_months * 0.42), max(8, n_sites * 0.26)))

    mesh = ax.pcolormesh(x_edges, y_edges, data, cmap=CMAP, vmin=0.5, vmax=max(4, np.nanmax(counts.to_numpy())),
                          edgecolors="white", linewidth=0.6, zorder=2)

    # Heavier separator line at each city-group boundary, plus one city label
    # vertically centered on its group, pushed well clear of the SiteCode ticks.
    group_start = 0
    for i in range(1, n_sites + 1):
        at_boundary = i == n_sites or site_to_city[sites[i]] != site_to_city[sites[i - 1]]
        if not at_boundary:
            continue
        ax.axhline(group_start - 0.5, color=CITY_LINE_COLOR, linewidth=1.2, zorder=3)
        ax.text(-0.11, (group_start + i - 1) / 2, site_to_city[sites[group_start]],
                 transform=ax.get_yaxis_transform(), fontsize=16, color="#000000",
                 ha="right", va="center")
        group_start = i
    ax.axhline(n_sites - 0.5, color=CITY_LINE_COLOR, linewidth=1.2, zorder=3)

    # Dotted vertical line at each year boundary.
    for year in range(months[0].year, months[-1].year + 1):
        jan1 = pd.Timestamp(year=year, month=1, day=1)
        if months[0].to_timestamp() <= jan1 <= (months[-1] + 1).to_timestamp():
            ax.axvline(mdates.date2num(jan1), color=YEAR_LINE_COLOR, linewidth=1.0,
                       linestyle=(0, (3, 3)), zorder=4)

    ax.set_yticks(range(n_sites))
    ax.set_yticklabels(sites, fontsize=12)
    ax.tick_params(axis="x", length=0, labelsize=16)
    ax.set_ylim(n_sites - 0.5, -0.5)

    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.set_xlim(x_edges[0], x_edges[-1])
    ax.set_xlabel("Date (Yearly)", fontsize=20)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.spines["bottom"].set_visible(False)
    ax.tick_params(axis="y", length=0)
    ax.tick_params(axis="x", length=0)

    cbar = fig.colorbar(mesh, ax=ax, orientation="horizontal", fraction=0.02, pad=0.07, aspect=40, shrink=0.25,
                         location="bottom", anchor=(0.0, 1.0))
    cbar.set_label("Sample dates per month", fontsize=16, color="#000000")
    cbar.ax.tick_params(labelsize=12)

    ax.set_title(
        "Samples collected per month by sampling site (2022-2026)",
        fontsize=24, fontweight="bold", loc="center", pad=16)

    fig.tight_layout(rect=[0.07, 0.03, 1, 1])

    if os.path.dirname(OUT_PATH):
        os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
    fig.savefig(OUT_PATH, dpi=300, bbox_inches="tight", pad_inches=0.3)
    fig.savefig(OUT_PATH.replace(".png", ".svg"), dpi=300, bbox_inches="tight", pad_inches=0.3)
    print(f"Saved {OUT_PATH}")


if __name__ == "__main__":
    main()

50 sites across 16 cities, 51 months (2022-05 to 2026-07)
Saved fig1b_site_sample_dates_Final.png
